# Create the Corpus Problem Lists

The notebook below creates the lists of documents for the corpus and datatype. This is used for iterating across the documents in the corpus when creating jobscripts that pull the known and unknown documents. To run this for a new set of data simply change the **corpus** and **data_type** parameters.

In [213]:
import sys
import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from utils import get_base_location, build_metadata_df, apply_temp_doc_id
from read_and_write_docs import read_jsonl, read_rds

In [214]:
corpus      = "Amazon"
data_type   = "training"

# Set NAS so can run on Windows laptop seamlessly
nas_base_loc = get_base_location()

# --- set your args (strings) ---
# nas_base_loc = "/Users/user/Documents/uni_work_offline"

known_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}/known_raw.jsonl"
unknown_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}/unknown_raw.jsonl"
metadata_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/metadata.rds"

save_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}"

## Read Data

In [215]:
metadata = read_rds(metadata_loc)
filtered_metadata = metadata[metadata['corpus'] == corpus]

known = read_jsonl(known_loc)
unknown = read_jsonl(unknown_loc)

In [216]:
filtered_metadata

,problem,corpus,known_author,unknown_author
186,A10AFVU66A79Y1 vs A10AFVU66A79Y1,Amazon,A10AFVU66A79Y1,A10AFVU66A79Y1
187,A10AFVU66A79Y1 vs A10ASLX7DTTB6Z,Amazon,A10AFVU66A79Y1,A10ASLX7DTTB6Z
188,A10ASLX7DTTB6Z vs A10ASLX7DTTB6Z,Amazon,A10ASLX7DTTB6Z,A10ASLX7DTTB6Z
189,A10ASLX7DTTB6Z vs A10C4O0Q0TWXOL,Amazon,A10ASLX7DTTB6Z,A10C4O0Q0TWXOL
190,A10C4O0Q0TWXOL vs A10C4O0Q0TWXOL,Amazon,A10C4O0Q0TWXOL,A10C4O0Q0TWXOL
...,...,...,...,...
1781,A2C8W1VCVWPY13 vs A2C9XE9I8RSKNX,Amazon,A2C8W1VCVWPY13,A2C9XE9I8RSKNX
1782,A2C9XE9I8RSKNX vs A2C9XE9I8RSKNX,Amazon,A2C9XE9I8RSKNX,A2C9XE9I8RSKNX
1783,A2C9XE9I8RSKNX vs A2CF66KIQ3RKX3,Amazon,A2C9XE9I8RSKNX,A2CF66KIQ3RKX3
1784,A2CF66KIQ3RKX3 vs A10AFVU66A79Y1,Amazon,A2CF66KIQ3RKX3,A10AFVU66A79Y1


## Create Agg Problem Level Metadata

In [217]:
agg_problems = filtered_metadata['problem'].drop_duplicates()

## Create Metadata

Quite a convoluted process.

In [218]:
# Build the dataframe
complete_metadata = build_metadata_df(filtered_metadata, known, unknown)

# Set blank text column for function to work
complete_metadata['text'] = ''

# Rename the known column and create the new doc_id
complete_metadata.rename(columns={"known_doc_id": "orig_doc_id"}, inplace=True)
complete_metadata = apply_temp_doc_id(complete_metadata)
complete_metadata.rename(columns={
    "orig_doc_id": "orig_known_doc_id",
    "doc_id": "known_doc_id",
    "unknown_doc_id": "orig_doc_id"
}, inplace=True)

# Do the same for the unknown
complete_metadata = apply_temp_doc_id(complete_metadata)
complete_metadata.rename(columns={
    "orig_doc_id": "orig_unknown_doc_id",
    "doc_id": "unknown_doc_id",
}, inplace=True)

# Sort columns
complete_metadata = complete_metadata[["sample_id", "problem", "corpus", "known_doc_id", "unknown_doc_id"]]

## View the data

In [219]:
complete_metadata.head()

,sample_id,problem,corpus,known_doc_id,unknown_doc_id
0,1,A10AFVU66A79Y1 vs A10AFVU66A79Y1,Amazon,a10afvu66a79y1_electronics,a10afvu66a79y1_cellphonesandaccessories
1,2,A10AFVU66A79Y1 vs A10AFVU66A79Y1,Amazon,a10afvu66a79y1_groceryandgourmetfood,a10afvu66a79y1_cellphonesandaccessories
2,3,A10AFVU66A79Y1 vs A10AFVU66A79Y1,Amazon,a10afvu66a79y1_healthandpersonalcare,a10afvu66a79y1_cellphonesandaccessories
3,4,A10AFVU66A79Y1 vs A10AFVU66A79Y1,Amazon,a10afvu66a79y1_homeandkitchen,a10afvu66a79y1_cellphonesandaccessories
4,5,A10AFVU66A79Y1 vs A10ASLX7DTTB6Z,Amazon,a10afvu66a79y1_electronics,a10aslx7dttb6z_baby


## Create the Document Lists

In [220]:
known_doc_list = pd.Series(complete_metadata["known_doc_id"].astype(str))
unknown_doc_list = pd.Series(complete_metadata["unknown_doc_id"].astype(str))
problem_doc_list = known_doc_list + ' vs ' + unknown_doc_list

## Get Number of Rows in the Dataset

This is used for the jobscript.

In [221]:
num_rows_for_jobscript = complete_metadata.shape[0]
print(f"Number of rows needed in jobscript: {num_rows_for_jobscript}")

Number of rows needed in jobscript: 6400


## Save the Lists

In [222]:
print(f"Writing agg problem list - {len(agg_problems)} Aggregated Problems")
agg_problems.to_csv(f"{save_loc}/agg_problem_list.txt", index=False, header=False)
print(f"Writing problem doc list - {len(problem_doc_list)} Problems")
problem_doc_list.to_csv(f"{save_loc}/problem_doc_list.txt", index=False, header=False)
print(f"Writing known doc list - {len(known_doc_list)} Documents")
known_doc_list.to_csv(f"{save_loc}/known_doc_list.txt", index=False, header=False)
print(f"Writing unknown doc list - {len(unknown_doc_list)} Documents")
unknown_doc_list.to_csv(f"{save_loc}/unknown_doc_list.txt", index=False, header=False)
print("Wrote doc lists")

Writing agg problem list - 1600 Aggregated Problems
Writing problem doc list - 6400 Problems
Writing known doc list - 6400 Documents
Writing unknown doc list - 6400 Documents
Wrote doc lists
